Import & configure

In [17]:
# ================================================================
# NOTEBOOK 9 — CELL 2
# IMPORTS + CONFIGURATION
# ================================================================

import pandas as pd
import numpy as np
import joblib
import torch
import torch.nn as nn

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# ================================================================
# PATHS
# ================================================================

BASE_DIR = Path(r"E:\ML_Project")

RESULTS_DIR = BASE_DIR / "results"
ORACLE_DIR = RESULTS_DIR / "oracle_labels"

# Existing validated Oracle labels
ORACLE_LABEL_FILE = (
    ORACLE_DIR /
    "oracle_labels_00000_70000.csv"
)

# Existing validated IQA training features
IQA_FILE = (
    RESULTS_DIR /
    "iqa_final_features_train.csv"
)

# MLP output directory
MLP_DIR = (
    RESULTS_DIR /
    "mlp_policy"
)

MLP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ================================================================
# CONFIGURATION
# ================================================================

RANDOM_STATE = 42

VALIDATION_SIZE = 0.15

# ================================================================
# 13 IQA FEATURES
# ================================================================

IQA_FEATURES = [
    "mean_brightness",
    "rms_contrast",
    "laplacian_variance",
    "tenengrad",
    "entropy",
    "dark_pixel_ratio",
    "bright_pixel_ratio",
    "mean_saturation",
    "saturation_std",
    "colorfulness",
    "edge_density",
    "noise_proxy",
    "motion_blur_proxy"
]

# ================================================================
# ORACLE TARGET
# ================================================================

TARGET_COLUMN = "oracle_action_id"

ACTION_NAMES = {
    0: "Identity",
    1: "CLAHE",
    2: "Gamma",
    3: "Dehazing",
    4: "Denoising"
}

print("=" * 80)
print("NOTEBOOK 9 — MLP POLICY DATASET PREPARATION")
print("=" * 80)

print()
print("INPUTS")
print("-" * 80)

print(f"Oracle labels : {ORACLE_LABEL_FILE}")
print(f"IQA features  : {IQA_FILE}")

print()
print(f"IQA features  : {len(IQA_FEATURES)}")
print(f"Target        : {TARGET_COLUMN}")
print(f"Validation    : {VALIDATION_SIZE:.0%}")
print(f"Random state  : {RANDOM_STATE}")

print()
print("✓ Configuration completed.")

NOTEBOOK 9 — MLP POLICY DATASET PREPARATION

INPUTS
--------------------------------------------------------------------------------
Oracle labels : E:\ML_Project\results\oracle_labels\oracle_labels_00000_70000.csv
IQA features  : E:\ML_Project\results\iqa_final_features_train.csv

IQA features  : 13
Target        : oracle_action_id
Validation    : 15%
Random state  : 42

✓ Configuration completed.


Verify Oracle Target Construction

In [12]:
# ================================================================
# CELL 6 — VERIFY ORACLE TARGET FROM EXISTING REWARD COLUMNS
# ================================================================

print("=" * 80)
print("VERIFYING ORACLE TARGET CONSTRUCTION")
print("=" * 80)

# ================================================================
# REWARD COLUMNS
# ================================================================

REWARD_COLUMNS = [
    "identity_reward",
    "clahe_reward",
    "gamma_reward",
    "dehazing_reward",
    "denoising_reward"
]

ACTION_ID_FROM_REWARD = {
    "identity_reward": 0,
    "clahe_reward": 1,
    "gamma_reward": 2,
    "dehazing_reward": 3,
    "denoising_reward": 4
}

ACTION_NAME_FROM_ID = {
    0: "Identity",
    1: "CLAHE",
    2: "Gamma",
    3: "Dehazing",
    4: "Denoising"
}

# ================================================================
# CHECK REQUIRED COLUMNS
# ================================================================

missing_reward_columns = [
    col
    for col in REWARD_COLUMNS
    if col not in policy_train.columns
]

if missing_reward_columns:

    raise KeyError(
        f"Missing reward columns: {missing_reward_columns}"
    )

print("✓ All five reward columns are present.")

# ================================================================
# CHECK REWARD VALUES
# ================================================================

print("\nREWARD VALUE CHECK")
print("-" * 80)

print(
    policy_train[REWARD_COLUMNS]
    .describe()
)

# ================================================================
# FIND BEST REWARD
# ================================================================

reward_matrix = policy_train[
    REWARD_COLUMNS
].to_numpy()

best_reward = reward_matrix.max(
    axis=1
)

best_action_positions = reward_matrix.argmax(
    axis=1
)

best_action_ids = best_action_positions.astype(
    int
)

best_action_names = [
    ACTION_NAME_FROM_ID[action_id]
    for action_id in best_action_ids
]

# ================================================================
# ADD TEMPORARY TARGETS
# ================================================================

policy_train_check = policy_train[
    [
        "image",
        "identity_f1"
    ] +
    REWARD_COLUMNS
].copy()

policy_train_check[
    "derived_oracle_action_id"
] = best_action_ids

policy_train_check[
    "derived_oracle_action"
] = best_action_names

policy_train_check[
    "derived_best_reward"
] = best_reward

# ================================================================
# ACTION DISTRIBUTION
# ================================================================

print("\nDERIVED ORACLE ACTION DISTRIBUTION")
print("-" * 80)

action_distribution = (
    policy_train_check[
        "derived_oracle_action_id"
    ]
    .value_counts()
    .sort_index()
)

for action_id, count in action_distribution.items():

    print(
        f"{action_id} "
        f"({ACTION_NAME_FROM_ID[action_id]:10s}) : "
        f"{count:6,} "
        f"({count / len(policy_train) * 100:6.2f}%)"
    )

# ================================================================
# TIE CHECK
# ================================================================

number_of_best_actions = (
    np.isclose(
        reward_matrix,
        best_reward[:, None],
        atol=1e-10
    ).sum(axis=1)
)

tie_mask = (
    number_of_best_actions > 1
)

tie_count = int(
    tie_mask.sum()
)

print("\nORACLE TIE CHECK")
print("-" * 80)

print(
    f"Images with multiple equally-best actions : "
    f"{tie_count:,}"
)

print(
    f"Images with a unique best action          : "
    f"{len(policy_train) - tie_count:,}"
)

# ================================================================
# SHOW TIE EXAMPLES
# ================================================================

if tie_count > 0:

    print("\nEXAMPLES OF TIED ORACLE REWARDS")
    print("-" * 80)

    display(
        policy_train_check.loc[
            tie_mask
        ].head(10)
    )

print("\n" + "=" * 80)
print("CELL 6 COMPLETED")
print("=" * 80)

print()
print("Important:")
print("The MLP target must reproduce the ORIGINAL Oracle")
print("tie-breaking rule, not an arbitrary new argmax rule.")

VERIFYING ORACLE TARGET CONSTRUCTION
✓ All five reward columns are present.

REWARD VALUE CHECK
--------------------------------------------------------------------------------
       identity_reward  clahe_reward  gamma_reward  dehazing_reward  denoising_reward
count          70000.0  70000.000000  70000.000000     70000.000000      70000.000000
mean               0.0     -0.012607     -0.002843        -0.025353         -0.026053
std                0.0      0.068638      0.044683         0.075324          0.081904
min                0.0     -0.666667     -0.666667        -0.857143         -0.833333
25%                0.0     -0.045455     -0.016484        -0.064286         -0.068841
50%                0.0      0.000000      0.000000        -0.017460         -0.017070
75%                0.0      0.021677      0.011039         0.013062          0.018182
max                0.0      0.750000      0.550000         0.500000          0.571429

DERIVED ORACLE ACTION DISTRIBUTION
-------------

,image,identity_f1,identity_reward,clahe_reward,gamma_reward,dehazing_reward,denoising_reward,derived_oracle_action_id,derived_oracle_action,derived_best_reward
0,0000f77c-6257be58.jpg,0.923077,0.0,-0.065934,0.000000,-0.153846,0.000000,0,Identity,0.000000
1,0000f77c-62c2a288.jpg,0.500000,0.0,0.000000,-0.055556,0.000000,-0.250000,0,Identity,0.000000
2,0000f77c-cb820c98.jpg,0.875000,0.0,0.058333,0.125000,0.125000,0.125000,2,Gamma,0.125000
4,0001542f-7c670be8.jpg,0.769231,0.0,0.000000,0.000000,-0.054945,-0.102564,0,Identity,0.000000
6,0004974f-05e1c285.jpg,0.400000,0.0,0.044444,0.044444,0.000000,-0.400000,1,CLAHE,0.044444
7,00054602-3bf57337.jpg,0.714286,0.0,0.119048,0.054945,0.119048,0.000000,1,CLAHE,0.119048
13,0008a165-c48f4b3e.jpg,0.761905,0.0,0.038095,-0.034632,0.038095,0.000000,1,CLAHE,0.038095
16,00091078-84635cf2.jpg,0.666667,0.0,-0.030303,0.000000,-0.190476,-0.016667,0,Identity,0.000000
17,00091078-875c1f73.jpg,0.777778,0.0,-0.027778,0.000000,-0.027778,-0.071895,0,Identity,0.000000
18,00091078-c1d32eea.jpg,0.769231,0.0,0.000000,0.000000,-0.102564,-0.076923,0,Identity,0.000000



CELL 6 COMPLETED

Important:
The MLP target must reproduce the ORIGINAL Oracle
tie-breaking rule, not an arbitrary new argmax rule.


Attach Existing Oracle Labels

In [13]:
# ================================================================
# CELL 7 — ATTACH EXISTING VALIDATED ORACLE LABELS
# ================================================================

print("=" * 80)
print("ATTACHING EXISTING ORACLE LABELS")
print("=" * 80)

# ================================================================
# LOAD EXISTING ORACLE LABELS
# ================================================================

ORACLE_LABEL_FILE = (
    ORACLE_DIR /
    "oracle_labels_00000_70000.csv"
)

if not ORACLE_LABEL_FILE.exists():
    raise FileNotFoundError(
        f"Oracle label file not found:\n{ORACLE_LABEL_FILE}"
    )

oracle_labels_existing = pd.read_csv(
    ORACLE_LABEL_FILE
)

print(
    f"Oracle labels loaded : "
    f"{len(oracle_labels_existing):,}"
)

# ================================================================
# STANDARDIZE IMAGE COLUMN
# ================================================================

policy_train["image"] = (
    policy_train["image"]
    .astype(str)
)

oracle_labels_existing["image"] = (
    oracle_labels_existing["image"]
    .astype(str)
)

# ================================================================
# MERGE
# ================================================================

mlp_train = policy_train.merge(
    oracle_labels_existing[
        [
            "image",
            "oracle_action_id",
            "oracle_action",
            "oracle_f1"
        ]
    ],
    on="image",
    how="inner",
    validate="one_to_one"
)

# ================================================================
# CHECK
# ================================================================

print()
print(
    f"Policy training rows : "
    f"{len(policy_train):,}"
)

print(
    f"Oracle label rows    : "
    f"{len(oracle_labels_existing):,}"
)

print(
    f"Merged training rows : "
    f"{len(mlp_train):,}"
)

print(
    f"Unique images        : "
    f"{mlp_train['image'].nunique():,}"
)

assert len(mlp_train) == 70_000
assert mlp_train["image"].nunique() == 70_000

print()
print("✓ Existing Oracle labels successfully attached.")
print("✓ No new Oracle labels generated.")

ATTACHING EXISTING ORACLE LABELS
Oracle labels loaded : 70,000

Policy training rows : 70,000
Oracle label rows    : 70,000
Merged training rows : 70,000
Unique images        : 70,000

✓ Existing Oracle labels successfully attached.
✓ No new Oracle labels generated.


In [34]:
# ================================================================
# CELL 27 — BUILD VALIDATION GROUND-TRUTH REWARD TABLE
# ================================================================

print("=" * 80)
print("BUILDING VALIDATION GROUND-TRUTH REWARD TABLE")
print("=" * 80)

# ================================================================
# LOAD THE FIVE VALIDATION DETECTION RESULTS
# ================================================================

validation_results = {}

for action, path in VALIDATION_RESULT_FILES.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Missing validation result for {action}:\n{path}"
        )

    df = pd.read_csv(path)

    # Keep only what we need
    validation_results[action] = (
        df[
            [
                "image",
                "f1"
            ]
        ]
        .copy()
    )

    validation_results[action] = (
        validation_results[action]
        .rename(
            columns={
                "f1": f"{action.lower()}_f1"
            }
        )
    )

# ================================================================
# MERGE BY IMAGE
# ================================================================

validation_f1 = validation_results["Identity"]

for action in [
    "CLAHE",
    "Gamma",
    "Dehazing",
    "Denoising"
]:

    validation_f1 = validation_f1.merge(
        validation_results[action],
        on="image",
        how="inner",
        validate="one_to_one"
    )

# ================================================================
# CHECK ROWS
# ================================================================

print(
    f"Validation images after merge : "
    f"{len(validation_f1):,}"
)

assert len(validation_f1) == 10_000

assert (
    validation_f1["image"].nunique()
    == 10_000
)

# ================================================================
# CONVERT F1 → REWARD
# ================================================================

identity_f1 = (
    validation_f1["identity_f1"]
)

validation_f1["identity_reward"] = 0.0

validation_f1["clahe_reward"] = (
    validation_f1["clahe_f1"]
    - identity_f1
)

validation_f1["gamma_reward"] = (
    validation_f1["gamma_f1"]
    - identity_f1
)

validation_f1["dehazing_reward"] = (
    validation_f1["dehazing_f1"]
    - identity_f1
)

validation_f1["denoising_reward"] = (
    validation_f1["denoising_f1"]
    - identity_f1
)

# ================================================================
# FINAL REWARD TABLE
# ================================================================

VALIDATION_REWARD_COLUMNS = [
    "identity_reward",
    "clahe_reward",
    "gamma_reward",
    "dehazing_reward",
    "denoising_reward"
]

validation_rewards = validation_f1[
    [
        "image"
    ] +
    VALIDATION_REWARD_COLUMNS
].copy()

# ================================================================
# DISPLAY
# ================================================================

print("\nVALIDATION REWARD TABLE")
print("-" * 80)

print(
    f"Rows    : {len(validation_rewards):,}"
)

print(
    f"Columns : {len(validation_rewards.columns)}"
)

display(
    validation_rewards.head()
)

print("\nREWARD STATISTICS")
print("-" * 80)

display(
    validation_rewards[
        VALIDATION_REWARD_COLUMNS
    ]
    .describe()
    .round(6)
)

print()
print("=" * 80)
print("CELL 27 COMPLETED")
print("=" * 80)

print("✓ Five validation F1 tables merged.")
print("✓ Validation rewards calculated relative to Identity.")
print("✓ 10,000 validation images available.")

BUILDING VALIDATION GROUND-TRUTH REWARD TABLE
Validation images after merge : 10,000

VALIDATION REWARD TABLE
--------------------------------------------------------------------------------
Rows    : 10,000
Columns : 6


,image,identity_reward,clahe_reward,gamma_reward,dehazing_reward,denoising_reward
0,b1c66a42-6f7d68ca.jpg,0.0,0.000000,0.000000,-0.011494,0.000000
1,b1c81faa-3df17267.jpg,0.0,-0.083333,0.000000,-0.083333,-0.178571
2,b1c81faa-c80764c5.jpg,0.0,-0.063241,0.027668,0.000000,-0.154150
3,b1c9c847-3bda4659.jpg,0.0,-0.088589,0.000000,-0.021337,-0.033033
4,b1ca2e5d-84cf9134.jpg,0.0,0.017094,-0.038462,-0.038462,-0.058462



REWARD STATISTICS
--------------------------------------------------------------------------------


,identity_reward,clahe_reward,gamma_reward,dehazing_reward,denoising_reward
count,10000.0,10000.000000,10000.000000,10000.000000,10000.000000
mean,0.0,-0.011557,-0.004470,-0.016087,-0.064060
std,0.0,0.069392,0.048782,0.061865,0.100636
min,0.0,-0.666667,-0.484848,-0.750000,-0.750000
25%,0.0,-0.044118,-0.022177,-0.045324,-0.120879
50%,0.0,0.000000,0.000000,0.000000,-0.058018
75%,0.0,0.022222,0.013746,0.013767,0.000000
max,0.0,0.457143,0.400000,0.363636,0.550000



CELL 27 COMPLETED
✓ Five validation F1 tables merged.
✓ Validation rewards calculated relative to Identity.
✓ 10,000 validation images available.


In [41]:
# ================================================================
# CELL 31 — SCALE 12-FEATURE TRAINING DATA
# ================================================================

print("=" * 80)
print("STANDARDIZING 12 IQA FEATURES")
print("=" * 80)

reward_scaler = StandardScaler()

X_reward_scaled = reward_scaler.fit_transform(
    X_reward
)

print(
    f"Training shape : "
    f"{X_reward_scaled.shape}"
)

print(
    f"Mean after scaling : "
    f"{X_reward_scaled.mean():.6f}"
)

print(
    f"Std after scaling  : "
    f"{X_reward_scaled.std():.6f}"
)

assert X_reward_scaled.shape == (
    70_000,
    12
)

print()
print("✓ Scaler fitted on the 70K training data.")
print("✓ No validation information used.")

print()
print("=" * 80)
print("CELL 31 COMPLETED")
print("=" * 80)

STANDARDIZING 12 IQA FEATURES
Training shape : (70000, 12)
Mean after scaling : -0.000000
Std after scaling  : 1.000000

✓ Scaler fitted on the 70K training data.
✓ No validation information used.

CELL 31 COMPLETED


Define the 12 → 64 → 32 → 5 Reward Regression MLP

In [42]:
# ================================================================
# CELL 32 — DEFINE 12-FEATURE REWARD REGRESSION MLP
# ================================================================

print("=" * 80)
print("DEFINING REWARD REGRESSION MLP")
print("=" * 80)


class RewardRegressionMLP(nn.Module):

    def __init__(
        self,
        input_dim=12,
        hidden1=64,
        hidden2=32,
        output_dim=5
    ):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden1
            ),

            nn.ReLU(),

            nn.Linear(
                hidden1,
                hidden2
            ),

            nn.ReLU(),

            nn.Linear(
                hidden2,
                output_dim
            )
        )

    def forward(self, x):
        return self.network(x)


# ================================================================
# CREATE MODEL
# ================================================================

reward_model = RewardRegressionMLP(
    input_dim=12,
    hidden1=64,
    hidden2=32,
    output_dim=5
).to(device)


# ================================================================
# LOSS
# ================================================================

reward_criterion = nn.MSELoss()


# ================================================================
# OPTIMIZER
# ================================================================

reward_optimizer = torch.optim.Adam(
    reward_model.parameters(),
    lr=1e-3
)


# ================================================================
# DISPLAY
# ================================================================

print("\nMODEL ARCHITECTURE")
print("-" * 80)

print(reward_model)

print("\nLOSS")
print("-" * 80)
print("Mean Squared Error (MSE)")

print("\nACTIVATION")
print("-" * 80)
print("Hidden Layer 1 : ReLU")
print("Hidden Layer 2 : ReLU")
print("Output         : 5 continuous rewards")

print("\nOPTIMIZER")
print("-" * 80)
print("Adam")
print("Learning rate : 0.001")

print()
print("=" * 80)
print("CELL 32 COMPLETED")
print("=" * 80)

DEFINING REWARD REGRESSION MLP

MODEL ARCHITECTURE
--------------------------------------------------------------------------------
RewardRegressionMLP(
  (network): Sequential(
    (0): Linear(in_features=12, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=5, bias=True)
  )
)

LOSS
--------------------------------------------------------------------------------
Mean Squared Error (MSE)

ACTIVATION
--------------------------------------------------------------------------------
Hidden Layer 1 : ReLU
Hidden Layer 2 : ReLU
Output         : 5 continuous rewards

OPTIMIZER
--------------------------------------------------------------------------------
Adam
Learning rate : 0.001

CELL 32 COMPLETED


Dataloader

In [43]:
# ================================================================
# CELL 33 — CREATE REWARD REGRESSION DATALOADER
# ================================================================

print("=" * 80)
print("CREATING REWARD REGRESSION DATALOADER")
print("=" * 80)

X_reward_tensor = torch.tensor(
    X_reward_scaled,
    dtype=torch.float32
)

Y_reward_tensor = torch.tensor(
    Y_reward.to_numpy(),
    dtype=torch.float32
)

reward_dataset = TensorDataset(
    X_reward_tensor,
    Y_reward_tensor
)

REWARD_BATCH_SIZE = 256

reward_loader = DataLoader(
    reward_dataset,
    batch_size=REWARD_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(f"Samples    : {len(reward_dataset):,}")
print(f"Batch size : {REWARD_BATCH_SIZE}")
print(f"Batches    : {len(reward_loader):,}")

print()
print("✓ Reward regression DataLoader ready.")

print()
print("=" * 80)
print("CELL 33 COMPLETED")
print("=" * 80)

CREATING REWARD REGRESSION DATALOADER
Samples    : 70,000
Batch size : 256
Batches    : 274

✓ Reward regression DataLoader ready.

CELL 33 COMPLETED


Traning 

In [44]:
# ================================================================
# CELL 34 — TRAIN 12-FEATURE REWARD REGRESSION MLP
# ================================================================

print("=" * 80)
print("TRAINING 12-FEATURE REWARD REGRESSION MLP")
print("=" * 80)

REWARD_EPOCHS = 50

reward_history = {
    "epoch": [],
    "train_mse": []
}

for epoch in range(
    1,
    REWARD_EPOCHS + 1
):

    reward_model.train()

    running_loss = 0.0
    total_samples = 0

    for batch_X, batch_Y in reward_loader:

        batch_X = batch_X.to(
            device,
            non_blocking=True
        )

        batch_Y = batch_Y.to(
            device,
            non_blocking=True
        )

        reward_optimizer.zero_grad()

        predicted_rewards = reward_model(
            batch_X
        )

        loss = reward_criterion(
            predicted_rewards,
            batch_Y
        )

        loss.backward()

        reward_optimizer.step()

        n = batch_X.size(0)

        running_loss += (
            loss.item() * n
        )

        total_samples += n

    epoch_mse = (
        running_loss /
        total_samples
    )

    reward_history["epoch"].append(epoch)
    reward_history["train_mse"].append(epoch_mse)

    print(
        f"Epoch {epoch:02d}/{REWARD_EPOCHS} | "
        f"MSE: {epoch_mse:.8f}"
    )

print()
print("=" * 80)
print("12-FEATURE REWARD REGRESSION TRAINING COMPLETED")
print("=" * 80)

TRAINING 12-FEATURE REWARD REGRESSION MLP
Epoch 01/50 | MSE: 0.00447191
Epoch 02/50 | MSE: 0.00386891
Epoch 03/50 | MSE: 0.00383991
Epoch 04/50 | MSE: 0.00382228
Epoch 05/50 | MSE: 0.00381706
Epoch 06/50 | MSE: 0.00381062
Epoch 07/50 | MSE: 0.00380722
Epoch 08/50 | MSE: 0.00380524
Epoch 09/50 | MSE: 0.00380462
Epoch 10/50 | MSE: 0.00379966
Epoch 11/50 | MSE: 0.00379690
Epoch 12/50 | MSE: 0.00379608
Epoch 13/50 | MSE: 0.00379446
Epoch 14/50 | MSE: 0.00379412
Epoch 15/50 | MSE: 0.00379103
Epoch 16/50 | MSE: 0.00378890
Epoch 17/50 | MSE: 0.00379080
Epoch 18/50 | MSE: 0.00378981
Epoch 19/50 | MSE: 0.00378741
Epoch 20/50 | MSE: 0.00378775
Epoch 21/50 | MSE: 0.00378612
Epoch 22/50 | MSE: 0.00378424
Epoch 23/50 | MSE: 0.00378513
Epoch 24/50 | MSE: 0.00378480
Epoch 25/50 | MSE: 0.00378340
Epoch 26/50 | MSE: 0.00378026
Epoch 27/50 | MSE: 0.00378229
Epoch 28/50 | MSE: 0.00378130
Epoch 29/50 | MSE: 0.00377853
Epoch 30/50 | MSE: 0.00377892
Epoch 31/50 | MSE: 0.00377842
Epoch 32/50 | MSE: 0.0037767

In [46]:
# ================================================================
# CELL 35 — BUILD VALIDATION DATASET + EVALUATE POLICY
# ================================================================

print("=" * 80)
print("BUILDING 10K VALIDATION DATASET")
print("=" * 80)


# ================================================================
# LOAD EXISTING VALIDATION IQA
# ================================================================

IQA_VALID_FILE = (
    BASE_DIR /
    "results" /
    "iqa_final_features_valid.csv"
)

iqa_valid = pd.read_csv(
    IQA_VALID_FILE
)

# ------------------------------------------------
# Image identifier
# ------------------------------------------------

if "image_name" in iqa_valid.columns:
    iqa_valid["image"] = (
        iqa_valid["image_name"].astype(str)
    )
else:
    iqa_valid["image"] = (
        iqa_valid["image"].astype(str)
    )


# ================================================================
# BUILD VALIDATION REWARD TABLE
# ================================================================

validation_files = {
    "identity_reward":
        ORACLE_DIR /
        "identity_validation_detection_results.csv",

    "clahe_reward":
        ORACLE_DIR /
        "validation_fixed_baselines" /
        "clahe_validation_detection_results.csv",

    "gamma_reward":
        ORACLE_DIR /
        "validation_fixed_baselines" /
        "gamma_validation_detection_results.csv",

    "dehazing_reward":
        ORACLE_DIR /
        "validation_fixed_baselines" /
        "dehazing_validation_detection_results.csv",

    "denoising_reward":
        ORACLE_DIR /
        "denoising_validation_detection_results_gpu.csv"
}


# ------------------------------------------------
# Load F1 values
# ------------------------------------------------

f1_tables = {}

for reward_name, file_path in validation_files.items():

    df = pd.read_csv(file_path)

    f1_tables[reward_name] = (
        df[["image", "f1"]]
        .rename(
            columns={
                "f1": reward_name
            }
        )
    )


# ================================================================
# MERGE FIVE ACTION RESULTS
# ================================================================

validation_rewards = f1_tables[
    "identity_reward"
]

for reward_name in [
    "clahe_reward",
    "gamma_reward",
    "dehazing_reward",
    "denoising_reward"
]:

    validation_rewards = validation_rewards.merge(
        f1_tables[reward_name],
        on="image",
        how="inner",
        validate="one_to_one"
    )


# ================================================================
# CONVERT F1 → REWARD
# ================================================================

identity_f1 = validation_rewards[
    "identity_reward"
].copy()

for reward_name in [
    "clahe_reward",
    "gamma_reward",
    "dehazing_reward",
    "denoising_reward"
]:

    validation_rewards[reward_name] = (
        validation_rewards[reward_name]
        - identity_f1
    )

validation_rewards["identity_reward"] = 0.0


# ================================================================
# MERGE IQA + REWARDS
# ================================================================

validation_dataset = (
    iqa_valid[
        ["image"] + MLP_FEATURES
    ]
    .merge(
        validation_rewards,
        on="image",
        how="inner",
        validate="one_to_one"
    )
)


# ================================================================
# VERIFY
# ================================================================

print(
    f"Validation IQA rows : {len(iqa_valid):,}"
)

print(
    f"Validation reward rows : "
    f"{len(validation_rewards):,}"
)

print(
    f"Final merged rows : "
    f"{len(validation_dataset):,}"
)

assert len(validation_dataset) == 10_000

print()
print("✓ 10K validation IQA + actual F1 rewards merged.")


# ================================================================
# VALIDATION FEATURES
# ================================================================

X_valid = validation_dataset[
    MLP_FEATURES
].copy()


# ================================================================
# SCALE USING TRAINING SCALER
# ================================================================

X_valid_scaled = reward_scaler.transform(
    X_valid
)


# ================================================================
# PREDICT FIVE REWARDS
# ================================================================

reward_model.eval()

with torch.no_grad():

    X_valid_tensor = torch.tensor(
        X_valid_scaled,
        dtype=torch.float32
    ).to(device)

    predicted_rewards = (
        reward_model(
            X_valid_tensor
        )
        .cpu()
        .numpy()
    )


# ================================================================
# SELECT ACTION
# ================================================================

predicted_action = np.argmax(
    predicted_rewards,
    axis=1
)


# ================================================================
# ACTUAL REWARD MATRIX
# ================================================================

VALIDATION_REWARD_COLUMNS = [
    "identity_reward",
    "clahe_reward",
    "gamma_reward",
    "dehazing_reward",
    "denoising_reward"
]

actual_rewards = validation_dataset[
    VALIDATION_REWARD_COLUMNS
].to_numpy()


# ================================================================
# ACTUAL REWARD OBTAINED BY OUR POLICY
# ================================================================

row_indices = np.arange(
    len(actual_rewards)
)

policy_reward = actual_rewards[
    row_indices,
    predicted_action
]


# ================================================================
# IDENTITY BASELINE
# ================================================================

identity_reward = actual_rewards[:, 0]


# ================================================================
# ORACLE UPPER BOUND
# ================================================================

oracle_reward = actual_rewards.max(
    axis=1
)


# ================================================================
# MEAN PERFORMANCE
# ================================================================

identity_mean = identity_reward.mean()
policy_mean = policy_reward.mean()
oracle_mean = oracle_reward.mean()

policy_gain = (
    policy_mean -
    identity_mean
)

oracle_gain = (
    oracle_mean -
    identity_mean
)

gain_captured = (
    policy_gain /
    oracle_gain *
    100
    if oracle_gain != 0
    else np.nan
)


# ================================================================
# RESULTS
# ================================================================

print()
print("=" * 80)
print("FINAL 10K VALIDATION POLICY PERFORMANCE")
print("=" * 80)

print(
    f"\nIdentity mean reward : "
    f"{identity_mean:.6f}"
)

print(
    f"Policy mean reward   : "
    f"{policy_mean:.6f}"
)

print(
    f"Oracle mean reward   : "
    f"{oracle_mean:.6f}"
)

print(
    f"\nPolicy gain over Identity : "
    f"{policy_gain:.6f}"
)

print(
    f"Oracle gain over Identity : "
    f"{oracle_gain:.6f}"
)

print(
    f"Oracle gain captured      : "
    f"{gain_captured:.2f}%"
)


# ================================================================
# ACTION DISTRIBUTION
# ================================================================

print("\nPREDICTED ACTION DISTRIBUTION")
print("-" * 80)

for action_id in range(5):

    count = np.sum(
        predicted_action == action_id
    )

    percentage = (
        count /
        len(predicted_action)
        * 100
    )

    print(
        f"{action_id} "
        f"({ACTION_NAMES[action_id]:10s}) : "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )


# ================================================================
# VALIDATION ORACLE DISTRIBUTION
# ================================================================

oracle_action = np.argmax(
    actual_rewards,
    axis=1
)

print("\nVALIDATION ORACLE ACTION DISTRIBUTION")
print("-" * 80)

for action_id in range(5):

    count = np.sum(
        oracle_action == action_id
    )

    percentage = (
        count /
        len(oracle_action)
        * 100
    )

    print(
        f"{action_id} "
        f"({ACTION_NAMES[action_id]:10s}) : "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )


print()
print("=" * 80)
print("CELL 35 COMPLETED")
print("=" * 80)

BUILDING 10K VALIDATION DATASET
Validation IQA rows : 10,000
Validation reward rows : 10,000
Final merged rows : 10,000

✓ 10K validation IQA + actual F1 rewards merged.

FINAL 10K VALIDATION POLICY PERFORMANCE

Identity mean reward : 0.000000
Policy mean reward   : -0.000733
Oracle mean reward   : 0.036502

Policy gain over Identity : -0.000733
Oracle gain over Identity : 0.036502
Oracle gain captured      : -2.01%

PREDICTED ACTION DISTRIBUTION
--------------------------------------------------------------------------------
0 (Identity  ) :  7,703 ( 77.03%)
1 (CLAHE     ) :     51 (  0.51%)
2 (Gamma     ) :  2,231 ( 22.31%)
3 (Dehazing  ) :      3 (  0.03%)
4 (Denoising ) :     12 (  0.12%)

VALIDATION ORACLE ACTION DISTRIBUTION
--------------------------------------------------------------------------------
0 (Identity  ) :  3,900 ( 39.00%)
1 (CLAHE     ) :  2,358 ( 23.58%)
2 (Gamma     ) :  1,473 ( 14.73%)
3 (Dehazing  ) :  1,223 ( 12.23%)
4 (Denoising ) :  1,046 ( 10.46%)

CELL 35

In [5]:
# ================================================================
# CELL 36 — REBUILD + EVALUATE + STRATIFIED ANALYSIS
# ================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler


print("=" * 80)
print("REBUILDING 12-FEATURE REWARD POLICY")
print("=" * 80)


# ================================================================
# PATHS — DIRECT, NO BASE_DIR
# ================================================================

RESULTS_DIR = r"E:\ML_Project\results"
ORACLE_DIR = r"E:\ML_Project\results\oracle_labels"

TRAIN_FILE = (
    RESULTS_DIR +
    r"\oracle_labels\policy_train_00000_70000.csv"
)

IQA_VALID_FILE = (
    RESULTS_DIR +
    r"\iqa_final_features_valid.csv"
)


# ================================================================
# 1. LOAD TRAINING POLICY DATA
# ================================================================

policy_train = pd.read_csv(
    TRAIN_FILE
)

print()
print(
    f"Training policy rows : "
    f"{len(policy_train):,}"
)


# ================================================================
# 2. COMMON 12 IQA FEATURES
# ================================================================

MLP_FEATURES = [
    "mean_brightness",
    "rms_contrast",
    "laplacian_variance",
    "tenengrad",
    "entropy",
    "dark_pixel_ratio",
    "bright_pixel_ratio",
    "mean_saturation",
    "saturation_std",
    "colorfulness",
    "edge_density",
    "noise_proxy"
]

REWARD_COLUMNS = [
    "identity_reward",
    "clahe_reward",
    "gamma_reward",
    "dehazing_reward",
    "denoising_reward"
]


# ================================================================
# 3. TRAINING FEATURES + TARGETS
# ================================================================

X_train = policy_train[
    MLP_FEATURES
].copy()

Y_train = policy_train[
    REWARD_COLUMNS
].copy()


# ================================================================
# 4. SCALE TRAINING FEATURES
# ================================================================

reward_scaler = StandardScaler()

X_train_scaled = reward_scaler.fit_transform(
    X_train
)


# ================================================================
# 5. DEFINE MODEL
# ================================================================

class RewardRegressionMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(12, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 5)
        )

    def forward(self, x):
        return self.network(x)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print()
print(f"Device : {device}")


reward_model = RewardRegressionMLP().to(device)


# ================================================================
# 6. TRAINING SETUP
# ================================================================

X_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

Y_tensor = torch.tensor(
    Y_train.to_numpy(),
    dtype=torch.float32
)

dataset = TensorDataset(
    X_tensor,
    Y_tensor
)

loader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    reward_model.parameters(),
    lr=0.001
)


# ================================================================
# 7. TRAIN
# ================================================================

print()
print("=" * 80)
print("TRAINING REWARD REGRESSION MLP")
print("=" * 80)

EPOCHS = 50

for epoch in range(1, EPOCHS + 1):

    reward_model.train()

    total_loss = 0.0
    total_samples = 0

    for batch_X, batch_Y in loader:

        batch_X = batch_X.to(
            device,
            non_blocking=True
        )

        batch_Y = batch_Y.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        prediction = reward_model(
            batch_X
        )

        loss = criterion(
            prediction,
            batch_Y
        )

        loss.backward()

        optimizer.step()

        n = batch_X.size(0)

        total_loss += (
            loss.item() * n
        )

        total_samples += n

    mse = (
        total_loss /
        total_samples
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"MSE: {mse:.8f}"
    )


# ================================================================
# 8. LOAD EXISTING 10K VALIDATION IQA
# ================================================================

print()
print("=" * 80)
print("LOADING EXISTING 10K VALIDATION DATA")
print("=" * 80)

iqa_valid = pd.read_csv(
    IQA_VALID_FILE
)

if "image_name" in iqa_valid.columns:

    iqa_valid["image"] = (
        iqa_valid["image_name"]
        .astype(str)
    )

else:

    iqa_valid["image"] = (
        iqa_valid["image"]
        .astype(str)
    )


# ================================================================
# 9. LOAD EXISTING VALIDATION DETECTION RESULTS
# ================================================================

validation_files = {
    "identity_reward":
        ORACLE_DIR +
        r"\identity_validation_detection_results.csv",

    "clahe_reward":
        ORACLE_DIR +
        r"\validation_fixed_baselines\clahe_validation_detection_results.csv",

    "gamma_reward":
        ORACLE_DIR +
        r"\validation_fixed_baselines\gamma_validation_detection_results.csv",

    "dehazing_reward":
        ORACLE_DIR +
        r"\validation_fixed_baselines\dehazing_validation_detection_results.csv",

    "denoising_reward":
        ORACLE_DIR +
        r"\denoising_validation_detection_results_gpu.csv"
}


validation_rewards = None

for reward_name, file_path in validation_files.items():

    df = pd.read_csv(file_path)

    temp = (
        df[
            ["image", "f1"]
        ]
        .rename(
            columns={
                "f1": reward_name
            }
        )
    )

    if validation_rewards is None:

        validation_rewards = temp

    else:

        validation_rewards = validation_rewards.merge(
            temp,
            on="image",
            how="inner",
            validate="one_to_one"
        )


# ================================================================
# 10. CONVERT F1 → REWARD RELATIVE TO IDENTITY
# ================================================================

identity_f1 = validation_rewards[
    "identity_reward"
].copy()

for reward_name in [
    "clahe_reward",
    "gamma_reward",
    "dehazing_reward",
    "denoising_reward"
]:

    validation_rewards[reward_name] = (
        validation_rewards[reward_name]
        - identity_f1
    )

validation_rewards["identity_reward"] = 0.0


# ================================================================
# 11. MERGE IQA + VALIDATION REWARDS
# ================================================================

df_val = (
    iqa_valid[
        ["image"] + MLP_FEATURES
    ]
    .merge(
        validation_rewards,
        on="image",
        how="inner",
        validate="one_to_one"
    )
)

assert len(df_val) == 10_000

print(
    f"Validation rows : "
    f"{len(df_val):,}"
)


# ================================================================
# 12. VALIDATION PREDICTION
# ================================================================

X_val_scaled = reward_scaler.transform(
    df_val[MLP_FEATURES]
)

reward_model.eval()

with torch.no_grad():

    X_val_tensor = torch.tensor(
        X_val_scaled,
        dtype=torch.float32
    ).to(device)

    predicted_rewards = (
        reward_model(
            X_val_tensor
        )
        .cpu()
        .numpy()
    )


# ================================================================
# 13. SELECT ACTION
# ================================================================

predicted_action = np.argmax(
    predicted_rewards,
    axis=1
)


# ================================================================
# 14. ACTUAL REWARD OF SELECTED ACTION
# ================================================================

actual_rewards = df_val[
    REWARD_COLUMNS
].to_numpy()

policy_reward = actual_rewards[
    np.arange(len(df_val)),
    predicted_action
]

df_val["predicted_action_id"] = (
    predicted_action
)

df_val["policy_reward"] = (
    policy_reward
)

df_val["oracle_reward"] = (
    actual_rewards.max(axis=1)
)

df_val["oracle_gain"] = (
    df_val["oracle_reward"]
)


# ================================================================
# 15. GAIN BUCKETS
# ================================================================

bins = [
    -0.01,
     0.01,
     0.05,
     0.10,
     1.00
]

labels = [
    "no_gain_available",
    "small_gain",
    "medium_gain",
    "large_gain"
]

df_val["gain_bucket"] = pd.cut(
    df_val["oracle_gain"],
    bins=bins,
    labels=labels,
    include_lowest=True
)


# ================================================================
# 16. STRATIFIED SUMMARY
# ================================================================

summary = (
    df_val
    .groupby(
        "gain_bucket",
        observed=False
    )
    .agg(
        n=("oracle_gain", "count"),

        policy_reward=(
            "policy_reward",
            "mean"
        ),

        oracle_reward=(
            "oracle_reward",
            "mean"
        ),

        available_gain=(
            "oracle_gain",
            "mean"
        )
    )
)

summary["policy_gain"] = (
    summary["policy_reward"]
)

summary["gain_captured_percent"] = np.where(
    summary["available_gain"] > 0,

    summary["policy_gain"]
    /
    summary["available_gain"]
    *
    100,

    np.nan
)


# ================================================================
# 17. PRINT STRATIFIED RESULTS
# ================================================================

print()
print("=" * 80)
print("STRATIFIED POLICY PERFORMANCE")
print("=" * 80)

print(
    summary.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)


# ================================================================
# 18. LARGE-GAIN CASE
# ================================================================

large = df_val[
    df_val["gain_bucket"] ==
    "large_gain"
]

print()
print("=" * 80)
print("LARGE-GAIN CASE")
print("=" * 80)

print(
    f"Images : {len(large):,}"
)

if len(large) > 0:

    large_policy = (
        large["policy_reward"].mean()
    )

    large_oracle = (
        large["oracle_reward"].mean()
    )

    print(
        f"Policy reward : "
        f"{large_policy:.6f}"
    )

    print(
        f"Oracle reward : "
        f"{large_oracle:.6f}"
    )

    print(
        f"Gain captured : "
        f"{large_policy / large_oracle * 100:.2f}%"
    )


# ================================================================
# 19. SAVE PREDICTIONS
# ================================================================

OUTPUT_FILE = (
    RESULTS_DIR +
    r"\final_analysis\reward_policy_validation_predictions.csv"
)

save_df = df_val[
    [
        "image",
        "predicted_action_id",
        "policy_reward",
        "oracle_reward",
        "oracle_gain",
        "gain_bucket"
    ]
].copy()

save_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print()
print(
    f"✓ Predictions saved to:\n"
    f"{OUTPUT_FILE}"
)


# ================================================================
# FINAL
# ================================================================

print()
print("=" * 80)
print("CELL 36 COMPLETED")
print("=" * 80)

REBUILDING 12-FEATURE REWARD POLICY

Training policy rows : 70,000

Device : cuda

TRAINING REWARD REGRESSION MLP
Epoch 01/50 | MSE: 0.00497188
Epoch 02/50 | MSE: 0.00387004
Epoch 03/50 | MSE: 0.00384664
Epoch 04/50 | MSE: 0.00383064
Epoch 05/50 | MSE: 0.00382513
Epoch 06/50 | MSE: 0.00381767
Epoch 07/50 | MSE: 0.00381330
Epoch 08/50 | MSE: 0.00380902
Epoch 09/50 | MSE: 0.00380691
Epoch 10/50 | MSE: 0.00380507
Epoch 11/50 | MSE: 0.00380252
Epoch 12/50 | MSE: 0.00380033
Epoch 13/50 | MSE: 0.00380026
Epoch 14/50 | MSE: 0.00379737
Epoch 15/50 | MSE: 0.00379720
Epoch 16/50 | MSE: 0.00379549
Epoch 17/50 | MSE: 0.00379323
Epoch 18/50 | MSE: 0.00379434
Epoch 19/50 | MSE: 0.00379283
Epoch 20/50 | MSE: 0.00379094
Epoch 21/50 | MSE: 0.00379072
Epoch 22/50 | MSE: 0.00379217
Epoch 23/50 | MSE: 0.00379031
Epoch 24/50 | MSE: 0.00379048
Epoch 25/50 | MSE: 0.00378740
Epoch 26/50 | MSE: 0.00378803
Epoch 27/50 | MSE: 0.00378649
Epoch 28/50 | MSE: 0.00378508
Epoch 29/50 | MSE: 0.00378511
Epoch 30/50 | MS

only inspect the metadata

In [8]:
# ================================================================
# CELL 38 — INSPECT EXISTING BDD100K METADATA
# ================================================================

import pandas as pd

METADATA_FILE = (
    r"E:\ML_Project\results\bdd100k_metadata.csv"
)

print("=" * 80)
print("INSPECTING EXISTING BDD100K METADATA")
print("=" * 80)


# ------------------------------------------------
# LOAD
# ------------------------------------------------

metadata_df = pd.read_csv(
    METADATA_FILE
)


# ------------------------------------------------
# BASIC STRUCTURE
# ------------------------------------------------

print()
print("DATASET STRUCTURE")
print("-" * 80)

print(
    f"Rows    : {len(metadata_df):,}"
)

print(
    f"Columns : {len(metadata_df.columns)}"
)


print()
print("COLUMNS")
print("-" * 80)

for i, column in enumerate(
    metadata_df.columns,
    1
):
    print(
        f"{i:2d}. {column}"
    )


# ------------------------------------------------
# DATA TYPES
# ------------------------------------------------

print()
print("DATA TYPES")
print("-" * 80)

print(
    metadata_df.dtypes
)


# ------------------------------------------------
# SAMPLE
# ------------------------------------------------

print()
print("FIRST 5 ROWS")
print("-" * 80)

print(
    metadata_df.head().to_string()
)


# ------------------------------------------------
# UNIQUE VALUES FOR CATEGORICAL COLUMNS
# ------------------------------------------------

print()
print("=" * 80)
print("CATEGORICAL / WEATHER-RELATED COLUMNS")
print("=" * 80)

for column in metadata_df.columns:

    # Skip high-cardinality columns
    unique_count = (
        metadata_df[column]
        .nunique(dropna=True)
    )

    if unique_count <= 30:

        print()
        print(
            f"{column}"
            f"  | unique values = "
            f"{unique_count}"
        )

        print(
            metadata_df[column]
            .value_counts(dropna=False)
            .head(30)
            .to_string()
        )


print()
print("=" * 80)
print("CELL 38 COMPLETED")
print("=" * 80)

INSPECTING EXISTING BDD100K METADATA

DATASET STRUCTURE
--------------------------------------------------------------------------------
Rows    : 100,000
Columns : 6

COLUMNS
--------------------------------------------------------------------------------
 1. image_name
 2. split
 3. weather
 4. scene
 5. timeofday
 6. num_objects

DATA TYPES
--------------------------------------------------------------------------------
image_name     object
split          object
weather        object
scene          object
timeofday      object
num_objects     int64
dtype: object

FIRST 5 ROWS
--------------------------------------------------------------------------------
          image_name  split weather        scene  timeofday  num_objects
0  0000f77c-6257be58  train   clear  city street    daytime            7
1  0000f77c-62c2a288  train   clear      highway  dawn/dusk            6
2  0000f77c-cb820c98  train   clear  residential  dawn/dusk            7
3  0001542f-5ce3cf52  train   clear  cit

In [9]:
# ================================================================
# CELL 39 — IDENTIFY EXISTING ADVERSE-WEATHER SUBSETS
# ================================================================

print("=" * 80)
print("IDENTIFYING EXISTING ADVERSE-WEATHER SUBSETS")
print("=" * 80)


# ================================================================
# ADVERSE WEATHER DEFINITION
# ================================================================

ADVERSE_WEATHER = [
    "rainy",
    "snowy",
    "foggy"
]


# ================================================================
# CHECK TRAINING SET
# ================================================================

train_weather = metadata_df[
    metadata_df["split"] == "train"
].copy()

train_adverse = train_weather[
    train_weather["weather"].isin(
        ADVERSE_WEATHER
    )
].copy()


# ================================================================
# CHECK VALIDATION SET
# ================================================================

valid_weather = metadata_df[
    metadata_df["split"] == "valid"
].copy()

valid_adverse = valid_weather[
    valid_weather["weather"].isin(
        ADVERSE_WEATHER
    )
].copy()


# ================================================================
# PRINT TRAINING DISTRIBUTION
# ================================================================

print()
print("TRAINING SET")
print("-" * 80)

print(
    f"Total training images : "
    f"{len(train_weather):,}"
)

print(
    f"Adverse-weather images : "
    f"{len(train_adverse):,}"
)

print(
    f"Adverse-weather ratio : "
    f"{len(train_adverse) / len(train_weather) * 100:.2f}%"
)

print()
print("Adverse-weather breakdown:")

print(
    train_adverse["weather"]
    .value_counts()
    .to_string()
)


# ================================================================
# PRINT VALIDATION DISTRIBUTION
# ================================================================

print()
print("VALIDATION SET")
print("-" * 80)

print(
    f"Total validation images : "
    f"{len(valid_weather):,}"
)

print(
    f"Adverse-weather images : "
    f"{len(valid_adverse):,}"
)

print(
    f"Adverse-weather ratio : "
    f"{len(valid_adverse) / len(valid_weather) * 100:.2f}%"
)

print()
print("Adverse-weather breakdown:")

print(
    valid_adverse["weather"]
    .value_counts()
    .to_string()
)


# ================================================================
# IMAGE NAME FORMAT CHECK
# ================================================================

print()
print("=" * 80)
print("IMAGE NAME FORMAT CHECK")
print("=" * 80)

print()
print("Metadata examples:")

print(
    train_adverse["image_name"]
    .head()
    .to_list()
)

print()
print("These will be matched against the existing")
print("Oracle/policy image names.")


# ================================================================
# SAVE SUBSET LISTS
# ================================================================

TRAIN_ADVERSE_FILE = (
    r"E:\ML_Project\results"
    r"\final_analysis"
    r"\adverse_weather_train_images.csv"
)

VALID_ADVERSE_FILE = (
    r"E:\ML_Project\results"
    r"\final_analysis"
    r"\adverse_weather_valid_images.csv"
)

train_adverse.to_csv(
    TRAIN_ADVERSE_FILE,
    index=False
)

valid_adverse.to_csv(
    VALID_ADVERSE_FILE,
    index=False
)


print()
print("=" * 80)
print("CELL 39 COMPLETED")
print("=" * 80)

print()
print(
    "✓ Existing weather metadata used."
)

print(
    "✓ No Oracle labels regenerated."
)

print(
    "✓ No YOLO inference performed."
)

print(
    "✓ Adverse-weather image lists saved."
)

IDENTIFYING EXISTING ADVERSE-WEATHER SUBSETS

TRAINING SET
--------------------------------------------------------------------------------
Total training images : 70,000
Adverse-weather images : 10,784
Adverse-weather ratio : 15.41%

Adverse-weather breakdown:
weather
snowy    5571
rainy    5083
foggy     130

VALIDATION SET
--------------------------------------------------------------------------------
Total validation images : 10,000
Adverse-weather images : 1,520
Adverse-weather ratio : 15.20%

Adverse-weather breakdown:
weather
snowy    769
rainy    738
foggy     13

IMAGE NAME FORMAT CHECK

Metadata examples:
['0004974f-05e1c285', '00091078-7cff8ea6', '00091078-84635cf2', '00091078-875c1f73', '00091078-c1d32eea']

These will be matched against the existing
Oracle/policy image names.

CELL 39 COMPLETED

✓ Existing weather metadata used.
✓ No Oracle labels regenerated.
✓ No YOLO inference performed.
✓ Adverse-weather image lists saved.


Analysis Trained Policy on the subset

In [10]:
# ================================================================
# CELL 40 — ADVERSE-WEATHER POLICY ANALYSIS
# ================================================================

print("=" * 80)
print("ADVERSE-WEATHER POLICY ANALYSIS")
print("=" * 80)


# ================================================================
# FILES
# ================================================================

PREDICTION_FILE = (
    r"E:\ML_Project\results"
    r"\final_analysis"
    r"\reward_policy_validation_predictions.csv"
)

WEATHER_FILE = (
    r"E:\ML_Project\results"
    r"\bdd100k_metadata.csv"
)


# ================================================================
# LOAD EXISTING POLICY PREDICTIONS
# ================================================================

policy_df = pd.read_csv(
    PREDICTION_FILE
)

metadata_df = pd.read_csv(
    WEATHER_FILE
)


# ================================================================
# NORMALIZE IMAGE NAMES
# ================================================================

policy_df["image_key"] = (
    policy_df["image"]
    .astype(str)
    .str.replace(
        ".jpg",
        "",
        regex=False
    )
)

metadata_df["image_key"] = (
    metadata_df["image_name"]
    .astype(str)
    .str.replace(
        ".jpg",
        "",
        regex=False
    )
)


# ================================================================
# SELECT VALIDATION + ADVERSE WEATHER
# ================================================================

adverse_weather = [
    "rainy",
    "snowy",
    "foggy"
]

valid_adverse = metadata_df[
    (
        metadata_df["split"] == "valid"
    )
    &
    (
        metadata_df["weather"].isin(
            adverse_weather
        )
    )
].copy()


# ================================================================
# MERGE WITH EXISTING POLICY RESULTS
# ================================================================

adverse_df = valid_adverse.merge(
    policy_df,
    on="image_key",
    how="inner",
    validate="one_to_one"
)


# ================================================================
# CHECK
# ================================================================

print()
print("MATCHING")
print("-" * 80)

print(
    f"Expected adverse-weather images : "
    f"{len(valid_adverse):,}"
)

print(
    f"Matched policy results           : "
    f"{len(adverse_df):,}"
)


if len(adverse_df) != len(valid_adverse):

    print()
    print("WARNING: Some images did not match.")

else:

    print(
        "✓ All adverse-weather validation "
        "images matched."
    )


# ================================================================
# OVERALL ADVERSE-WEATHER PERFORMANCE
# ================================================================

print()
print("=" * 80)
print("ADVERSE-WEATHER PERFORMANCE")
print("=" * 80)


identity_reward = 0.0

policy_reward = (
    adverse_df["policy_reward"]
    .mean()
)

oracle_reward = (
    adverse_df["oracle_reward"]
    .mean()
)


print()
print(
    f"Identity mean reward : "
    f"{identity_reward:.6f}"
)

print(
    f"Policy mean reward   : "
    f"{policy_reward:.6f}"
)

print(
    f"Oracle mean reward   : "
    f"{oracle_reward:.6f}"
)


policy_gain = (
    policy_reward -
    identity_reward
)

oracle_gain = (
    oracle_reward -
    identity_reward
)


print()
print(
    f"Policy gain over Identity : "
    f"{policy_gain:.6f}"
)

print(
    f"Oracle gain over Identity : "
    f"{oracle_gain:.6f}"
)


if oracle_gain > 0:

    captured = (
        policy_gain /
        oracle_gain *
        100
    )

    print(
        f"Oracle gain captured      : "
        f"{captured:.2f}%"
    )


# ================================================================
# WEATHER-BY-WEATHER ANALYSIS
# ================================================================

print()
print("=" * 80)
print("PERFORMANCE BY WEATHER TYPE")
print("=" * 80)


weather_summary = (
    adverse_df
    .groupby("weather")
    .agg(
        n=("image_key", "count"),

        policy_reward=(
            "policy_reward",
            "mean"
        ),

        oracle_reward=(
            "oracle_reward",
            "mean"
        )
    )
)


weather_summary["policy_gain"] = (
    weather_summary["policy_reward"]
)

weather_summary["oracle_gain"] = (
    weather_summary["oracle_reward"]
)

weather_summary["gain_captured_percent"] = np.where(
    weather_summary["oracle_gain"] > 0,

    weather_summary["policy_gain"]
    /
    weather_summary["oracle_gain"]
    *
    100,

    np.nan
)


print(
    weather_summary.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)


# ================================================================
# POLICY ACTION DISTRIBUTION
# ================================================================

print()
print("=" * 80)
print("MLP POLICY ACTION DISTRIBUTION")
print("=" * 80)

ACTION_NAMES = {
    0: "Identity",
    1: "CLAHE",
    2: "Gamma",
    3: "Dehazing",
    4: "Denoising"
}


action_counts = (
    adverse_df[
        "predicted_action_id"
    ]
    .value_counts()
    .sort_index()
)


for action_id in range(5):

    count = action_counts.get(
        action_id,
        0
    )

    percentage = (
        count /
        len(adverse_df) *
        100
    )

    print(
        f"{action_id} "
        f"({ACTION_NAMES[action_id]:10s}) : "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )


# ================================================================
# ORACLE ACTION DISTRIBUTION
# ================================================================

# The saved prediction file does not contain the Oracle action,
# so derive it from the available reward values if present.

oracle_action_available = all(
    column in adverse_df.columns
    for column in [
        "oracle_reward"
    ]
)

print()
print("=" * 80)
print("ADVERSE-WEATHER DATASET SUMMARY")
print("=" * 80)

print(
    f"Total adverse-weather validation images : "
    f"{len(adverse_df):,}"
)

print()

print(
    adverse_df["weather"]
    .value_counts()
    .to_string()
)


# ================================================================
# SAVE RESULT
# ================================================================

OUTPUT_FILE = (
    r"E:\ML_Project\results"
    r"\final_analysis"
    r"adverse_weather_policy_analysis_1520.csv"
)

adverse_df.to_csv(
    OUTPUT_FILE,
    index=False
)


print()
print("=" * 80)
print("CELL 40 COMPLETED")
print("=" * 80)

print()
print(
    f"✓ Adverse-weather analysis saved to:\n"
    f"{OUTPUT_FILE}"
)

ADVERSE-WEATHER POLICY ANALYSIS

MATCHING
--------------------------------------------------------------------------------
Expected adverse-weather images : 1,520
Matched policy results           : 1,520
✓ All adverse-weather validation images matched.

ADVERSE-WEATHER PERFORMANCE

Identity mean reward : 0.000000
Policy mean reward   : -0.002794
Oracle mean reward   : 0.036565

Policy gain over Identity : -0.002794
Oracle gain over Identity : 0.036565
Oracle gain captured      : -7.64%

PERFORMANCE BY WEATHER TYPE
           n  policy_reward  oracle_reward  policy_gain  oracle_gain  gain_captured_percent
weather                                                                                    
foggy     13      -0.006234       0.035866    -0.006234     0.035866             -17.382383
rainy    738      -0.002209       0.036611    -0.002209     0.036611              -6.032982
snowy    769      -0.003297       0.036533    -0.003297     0.036533              -9.024454

MLP POLICY ACTION D

In [11]:
# ================================================================
# CELL 41 — REWARD REGRESSION SANITY CHECK
# ================================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("REWARD REGRESSION SANITY CHECK")
print("=" * 80)


# ================================================================
# 1. TRAINING DATA
# ================================================================

TRAIN_FILE = (
    r"E:\ML_Project\results\oracle_labels"
    r"\policy_train_00000_70000.csv"
)

train_df = pd.read_csv(TRAIN_FILE)

print()
print("TRAINING DATA")
print("-" * 80)

print(
    f"Rows    : {len(train_df):,}"
)

print(
    f"Columns : {len(train_df.columns)}"
)


# ================================================================
# 2. IQA FEATURES
# ================================================================

IQA_FEATURES = [
    "mean_brightness",
    "rms_contrast",
    "laplacian_variance",
    "tenengrad",
    "entropy",
    "dark_pixel_ratio",
    "bright_pixel_ratio",
    "mean_saturation",
    "saturation_std",
    "colorfulness",
    "edge_density",
    "noise_proxy"
]


print()
print("=" * 80)
print("RAW IQA FEATURE SCALE")
print("=" * 80)

feature_stats = train_df[IQA_FEATURES].agg(
    ["mean", "std", "min", "max"]
).T

print(
    feature_stats.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)


# ================================================================
# 3. STANDARDIZE IQA FEATURES
# ================================================================

from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()

X_scaled = feature_scaler.fit_transform(
    train_df[IQA_FEATURES]
)


print()
print("=" * 80)
print("SCALED IQA FEATURE CHECK")
print("=" * 80)

scaled_mean = X_scaled.mean(axis=0)
scaled_std = X_scaled.std(axis=0)

scaled_check = pd.DataFrame({
    "feature": IQA_FEATURES,
    "mean_after_scaling": scaled_mean,
    "std_after_scaling": scaled_std
})

print(
    scaled_check.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ================================================================
# 4. REWARD TARGETS
# ================================================================

REWARD_COLS = [
    "identity_reward",
    "clahe_reward",
    "gamma_reward",
    "dehazing_reward",
    "denoising_reward"
]


print()
print("=" * 80)
print("RAW REWARD TARGET DISTRIBUTIONS")
print("=" * 80)

reward_stats = train_df[REWARD_COLS].agg(
    ["mean", "std", "min", "max"]
).T

print(
    reward_stats.to_string(
        float_format=lambda x: f"{x:.8f}"
    )
)


# ================================================================
# 5. REWARD VARIANCE
# ================================================================

print()
print("=" * 80)
print("REWARD TARGET VARIANCE")
print("=" * 80)

for col in REWARD_COLS:

    print(
        f"{col:20s} "
        f"std = {train_df[col].std():.8f}   "
        f"variance = {train_df[col].var():.8f}"
    )


# ================================================================
# 6. PER-IMAGE REWARD SPREAD
# ================================================================

reward_matrix = train_df[
    REWARD_COLS
].to_numpy()

best_reward = reward_matrix.max(axis=1)

worst_reward = reward_matrix.min(axis=1)

reward_spread = (
    best_reward -
    worst_reward
)

print()
print("=" * 80)
print("PER-IMAGE REWARD SPREAD")
print("=" * 80)

print(
    f"Mean spread : {reward_spread.mean():.8f}"
)

print(
    f"Median spread : {np.median(reward_spread):.8f}"
)

print(
    f"Max spread : {reward_spread.max():.8f}"
)


# ================================================================
# 7. HOW OFTEN IS THE BEST ACTION CLEARLY BETTER?
# ================================================================

print()
print("=" * 80)
print("ORACLE ACTION SEPARATION")
print("=" * 80)

thresholds = [
    0.001,
    0.005,
    0.010,
    0.020,
    0.050,
    0.100
]

for threshold in thresholds:

    count = np.sum(
        reward_spread >= threshold
    )

    percentage = (
        count /
        len(train_df) *
        100
    )

    print(
        f"Spread >= {threshold:.3f} : "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )


# ================================================================
# 8. CHECK REWARD TARGET STANDARDIZATION
# ================================================================

reward_scaler = StandardScaler()

Y_scaled = reward_scaler.fit_transform(
    train_df[REWARD_COLS]
)

scaled_reward_stats = pd.DataFrame(
    {
        "reward": REWARD_COLS,
        "mean_after_scaling": Y_scaled.mean(axis=0),
        "std_after_scaling": Y_scaled.std(axis=0)
    }
)

print()
print("=" * 80)
print("REWARD TARGETS AFTER STANDARDIZATION")
print("=" * 80)

print(
    scaled_reward_stats.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)


# ================================================================
# 9. IMPORTANT — COMPARE REWARD HEAD SCALE
# ================================================================

print()
print("=" * 80)
print("REWARD HEAD SCALE")
print("=" * 80)

for i, col in enumerate(REWARD_COLS):

    print(
        f"{i} ({col:20s}) "
        f"mean = {train_df[col].mean(): .8f}   "
        f"std = {train_df[col].std(): .8f}"
    )


# ================================================================
# 10. SUMMARY
# ================================================================

print()
print("=" * 80)
print("CELL 41 COMPLETED")
print("=" * 80)

print()
print("✓ IQA feature scaling inspected.")
print("✓ Reward target scaling inspected.")
print("✓ Per-image reward separation inspected.")
print("✓ No model retraining performed.")
print("✓ No Oracle labels regenerated.")
print("✓ Weather labels were not used as model features.")

REWARD REGRESSION SANITY CHECK

TRAINING DATA
--------------------------------------------------------------------------------
Rows    : 70,000
Columns : 20

RAW IQA FEATURE SCALE
                           mean         std        min          max
mean_brightness       73.479542   39.230667   0.215451   249.969025
rms_contrast          48.932288   16.959889   3.366528    92.744080
laplacian_variance  1048.608665  653.241123  12.621545 13682.840299
tenengrad          12449.495797 6478.823530 310.490694 84525.540208
entropy                6.799019    0.911924   0.444684     7.928956
dark_pixel_ratio       0.405114    0.319145   0.000000     0.999201
bright_pixel_ratio     0.045815    0.054021   0.000000     0.959010
mean_saturation       73.878340   34.665219   4.878819   234.448242
saturation_std        42.877638   15.437780   7.736240   122.537445
colorfulness          24.007369   14.171093   1.414865   137.576965
edge_density           0.083466    0.047642   0.001111     0.290729
nois

In [12]:
# ================================================================
# CELL 42 — TARGET-STANDARDIZED REWARD REGRESSION
# ================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

print("=" * 80)
print("TRAINING TARGET-STANDARDIZED REWARD REGRESSION MLP")
print("=" * 80)


# ================================================================
# 1. LOAD EXISTING 70K TRAINING POLICY DATA
# ================================================================

TRAIN_FILE = (
    r"E:\ML_Project\results\oracle_labels"
    r"\policy_train_00000_70000.csv"
)

train_df = pd.read_csv(TRAIN_FILE)

print()
print("Training rows :", f"{len(train_df):,}")


# ================================================================
# 2. FEATURES
# ================================================================

IQA_FEATURES = [
    "mean_brightness",
    "rms_contrast",
    "laplacian_variance",
    "tenengrad",
    "entropy",
    "dark_pixel_ratio",
    "bright_pixel_ratio",
    "mean_saturation",
    "saturation_std",
    "colorfulness",
    "edge_density",
    "noise_proxy"
]


REWARD_COLS = [
    "identity_reward",
    "clahe_reward",
    "gamma_reward",
    "dehazing_reward",
    "denoising_reward"
]


# ================================================================
# 3. SCALE IQA FEATURES
# ================================================================

feature_scaler = StandardScaler()

X = feature_scaler.fit_transform(
    train_df[IQA_FEATURES]
).astype(np.float32)


# ================================================================
# 4. STANDARDIZE NON-CONSTANT REWARD TARGETS
# ================================================================

# Identity reward is always exactly zero.
# Therefore it must NOT be passed through StandardScaler.

NON_IDENTITY_REWARDS = [
    "clahe_reward",
    "gamma_reward",
    "dehazing_reward",
    "denoising_reward"
]

reward_scaler = StandardScaler()

Y_non_identity = reward_scaler.fit_transform(
    train_df[NON_IDENTITY_REWARDS]
).astype(np.float32)


# ------------------------------------------------
# Reconstruct 5-output target matrix
# Identity remains zero.
# ------------------------------------------------

Y = np.zeros(
    (len(train_df), 5),
    dtype=np.float32
)

Y[:, 1:] = Y_non_identity


# ================================================================
# 5. CHECK TARGET SCALING
# ================================================================

print()
print("=" * 80)
print("STANDARDIZED REWARD TARGET CHECK")
print("=" * 80)

for i, col in enumerate(REWARD_COLS):

    print(
        f"{i} ({col:20s}) "
        f"mean = {Y[:, i].mean(): .6f}   "
        f"std = {Y[:, i].std(): .6f}"
    )


# ================================================================
# 6. DEVICE
# ================================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print()
print("Device :", device)


# ================================================================
# 7. TENSORS
# ================================================================

X_tensor = torch.tensor(
    X,
    dtype=torch.float32
)

Y_tensor = torch.tensor(
    Y,
    dtype=torch.float32
)


dataset = TensorDataset(
    X_tensor,
    Y_tensor
)

loader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True
)


# ================================================================
# 8. MODEL
# ================================================================

class RewardRegressionMLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(12, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 5)
        )

    def forward(self, x):

        return self.network(x)


model = RewardRegressionMLP().to(device)


# ================================================================
# 9. LOSS + OPTIMIZER
# ================================================================

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ================================================================
# 10. TRAIN
# ================================================================

EPOCHS = 50

print()
print("=" * 80)
print("TRAINING")
print("=" * 80)

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0

    for batch_X, batch_Y in loader:

        batch_X = batch_X.to(device)
        batch_Y = batch_Y.to(device)

        optimizer.zero_grad()

        predictions = model(batch_X)

        loss = criterion(
            predictions,
            batch_Y
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() *
            len(batch_X)
        )

    epoch_loss = (
        running_loss /
        len(dataset)
    )

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} "
        f"| MSE: {epoch_loss:.8f}"
    )


print()
print("=" * 80)
print("TARGET-STANDARDIZED TRAINING COMPLETED")
print("=" * 80)

TRAINING TARGET-STANDARDIZED REWARD REGRESSION MLP

Training rows : 70,000

STANDARDIZED REWARD TARGET CHECK
0 (identity_reward     ) mean =  0.000000   std =  0.000000
1 (clahe_reward        ) mean =  0.000000   std =  1.000000
2 (gamma_reward        ) mean = -0.000000   std =  1.000000
3 (dehazing_reward     ) mean = -0.000000   std =  1.000000
4 (denoising_reward    ) mean = -0.000000   std =  1.000000

Device : cuda

TRAINING
Epoch 01/50 | MSE: 0.79779499
Epoch 02/50 | MSE: 0.79637913
Epoch 03/50 | MSE: 0.79594693
Epoch 04/50 | MSE: 0.79572362
Epoch 05/50 | MSE: 0.79531433
Epoch 06/50 | MSE: 0.79515673
Epoch 07/50 | MSE: 0.79492395
Epoch 08/50 | MSE: 0.79471756
Epoch 09/50 | MSE: 0.79453586
Epoch 10/50 | MSE: 0.79427428
Epoch 11/50 | MSE: 0.79435823
Epoch 12/50 | MSE: 0.79412857
Epoch 13/50 | MSE: 0.79402906
Epoch 14/50 | MSE: 0.79371615
Epoch 15/50 | MSE: 0.79354499
Epoch 16/50 | MSE: 0.79349742
Epoch 17/50 | MSE: 0.79331407
Epoch 18/50 | MSE: 0.79321843
Epoch 19/50 | MSE: 0.79309

In [15]:
# ================================================================
# CELL 43 — EVALUATE TARGET-STANDARDIZED REWARD REGRESSION
#          USING ACTUAL 10K VALIDATION F1 REWARDS
# ================================================================

import numpy as np
import pandas as pd
import torch

print("=" * 80)
print("EVALUATING TARGET-STANDARDIZED REWARD REGRESSION")
print("=" * 80)


# ================================================================
# 1. FILE PATHS
# ================================================================

IQA_VALID_FILE = (
    r"E:\ML_Project\results"
    r"\iqa_final_features_valid.csv"
)

IDENTITY_FILE = (
    r"E:\ML_Project\results\oracle_labels"
    r"\identity_validation_detection_results.csv"
)

CLAHE_FILE = (
    r"E:\ML_Project\results\oracle_labels"
    r"\validation_fixed_baselines"
    r"\clahe_validation_detection_results.csv"
)

GAMMA_FILE = (
    r"E:\ML_Project\results\oracle_labels"
    r"\validation_fixed_baselines"
    r"\gamma_validation_detection_results.csv"
)

DEHAZING_FILE = (
    r"E:\ML_Project\results\oracle_labels"
    r"\validation_fixed_baselines"
    r"\dehazing_validation_detection_results.csv"
)

DENOISING_FILE = (
    r"E:\ML_Project\results\oracle_labels"
    r"\denoising_validation_detection_results_gpu.csv"
)


# ================================================================
# 2. LOAD VALIDATION IQA
# ================================================================

iqa_valid = pd.read_csv(
    IQA_VALID_FILE
)

print()
print(
    f"Validation IQA rows : "
    f"{len(iqa_valid):,}"
)


# ================================================================
# 3. NORMALIZE IQA IMAGE NAME
# ================================================================

if "image_name" not in iqa_valid.columns:

    raise KeyError(
        "Expected 'image_name' in validation IQA file."
    )


def normalize_image_name(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(
            "\\",
            "/",
            regex=False
        )
        .str.split("/")
        .str[-1]
        .str.replace(
            ".jpg",
            "",
            regex=False
        )
        .str.replace(
            ".jpeg",
            "",
            regex=False
        )
        .str.replace(
            ".png",
            "",
            regex=False
        )
    )


iqa_valid["image_key"] = normalize_image_name(
    iqa_valid["image_name"]
)


# ================================================================
# 4. LOAD EXISTING DETECTION RESULTS
# ================================================================

identity_df = pd.read_csv(
    IDENTITY_FILE
)

clahe_df = pd.read_csv(
    CLAHE_FILE
)

gamma_df = pd.read_csv(
    GAMMA_FILE
)

dehazing_df = pd.read_csv(
    DEHAZING_FILE
)

denoising_df = pd.read_csv(
    DENOISING_FILE
)


print()
print("=" * 80)
print("VALIDATION DETECTION RESULT FILES")
print("=" * 80)

print(
    f"Identity   : {len(identity_df):,}"
)

print(
    f"CLAHE      : {len(clahe_df):,}"
)

print(
    f"Gamma      : {len(gamma_df):,}"
)

print(
    f"Dehazing   : {len(dehazing_df):,}"
)

print(
    f"Denoising  : {len(denoising_df):,}"
)


# ================================================================
# 5. NORMALIZE IMAGE NAMES
# ================================================================

for df in [
    identity_df,
    clahe_df,
    gamma_df,
    dehazing_df,
    denoising_df
]:

    if "image" not in df.columns:

        raise KeyError(
            "A detection result file does not contain "
            "'image'."
        )

    df["image_key"] = normalize_image_name(
        df["image"]
    )


# ================================================================
# 6. CHECK F1 COLUMNS
# ================================================================

for name, df in [
    ("Identity", identity_df),
    ("CLAHE", clahe_df),
    ("Gamma", gamma_df),
    ("Dehazing", dehazing_df),
    ("Denoising", denoising_df)
]:

    if "f1" not in df.columns:

        raise KeyError(
            f"{name} detection result does not "
            "contain 'f1'."
        )


# ================================================================
# 7. BUILD ACTUAL VALIDATION F1 TABLE
# ================================================================

identity_rewards = identity_df[
    ["image_key", "f1"]
].rename(
    columns={
        "f1": "identity_f1"
    }
)

clahe_rewards = clahe_df[
    ["image_key", "f1"]
].rename(
    columns={
        "f1": "clahe_f1"
    }
)

gamma_rewards = gamma_df[
    ["image_key", "f1"]
].rename(
    columns={
        "f1": "gamma_f1"
    }
)

dehazing_rewards = dehazing_df[
    ["image_key", "f1"]
].rename(
    columns={
        "f1": "dehazing_f1"
    }
)

denoising_rewards = denoising_df[
    ["image_key", "f1"]
].rename(
    columns={
        "f1": "denoising_f1"
    }
)


# ================================================================
# 8. MERGE FIVE F1 TABLES
# ================================================================

actual_f1 = identity_rewards.merge(
    clahe_rewards,
    on="image_key",
    how="inner",
    validate="one_to_one"
)

actual_f1 = actual_f1.merge(
    gamma_rewards,
    on="image_key",
    how="inner",
    validate="one_to_one"
)

actual_f1 = actual_f1.merge(
    dehazing_rewards,
    on="image_key",
    how="inner",
    validate="one_to_one"
)

actual_f1 = actual_f1.merge(
    denoising_rewards,
    on="image_key",
    how="inner",
    validate="one_to_one"
)


print()
print("=" * 80)
print("ACTUAL VALIDATION F1 TABLE")
print("=" * 80)

print(
    f"Rows : {len(actual_f1):,}"
)


if len(actual_f1) != 10000:

    raise ValueError(
        "Expected 10,000 validation F1 rows, "
        f"got {len(actual_f1):,}"
    )


# ================================================================
# 9. CONSTRUCT REWARDS RELATIVE TO IDENTITY
# ================================================================

actual_f1["identity_reward"] = (
    actual_f1["identity_f1"]
    -
    actual_f1["identity_f1"]
)

actual_f1["clahe_reward"] = (
    actual_f1["clahe_f1"]
    -
    actual_f1["identity_f1"]
)

actual_f1["gamma_reward"] = (
    actual_f1["gamma_f1"]
    -
    actual_f1["identity_f1"]
)

actual_f1["dehazing_reward"] = (
    actual_f1["dehazing_f1"]
    -
    actual_f1["identity_f1"]
)

actual_f1["denoising_reward"] = (
    actual_f1["denoising_f1"]
    -
    actual_f1["identity_f1"]
)


REWARD_COLS = [
    "identity_reward",
    "clahe_reward",
    "gamma_reward",
    "dehazing_reward",
    "denoising_reward"
]


# ================================================================
# 10. MERGE IQA + ACTUAL REWARDS
# ================================================================

validation_dataset = iqa_valid[
    ["image_key"] + IQA_FEATURES
].merge(
    actual_f1[
        ["image_key"] + REWARD_COLS
    ],
    on="image_key",
    how="inner",
    validate="one_to_one"
)


print()
print("=" * 80)
print("FINAL VALIDATION DATASET")
print("=" * 80)

print(
    f"IQA rows              : "
    f"{len(iqa_valid):,}"
)

print(
    f"Actual F1 rows        : "
    f"{len(actual_f1):,}"
)

print(
    f"Final merged rows     : "
    f"{len(validation_dataset):,}"
)


if len(validation_dataset) != 10000:

    raise ValueError(
        "Final validation dataset must contain "
        "exactly 10,000 rows."
    )


print(
    "✓ Exact 10K validation dataset confirmed."
)


# ================================================================
# 11. SCALE VALIDATION IQA
# ================================================================

X_valid = feature_scaler.transform(
    validation_dataset[IQA_FEATURES]
).astype(np.float32)


X_valid_tensor = torch.tensor(
    X_valid,
    dtype=torch.float32
).to(device)


# ================================================================
# 12. PREDICT STANDARDIZED REWARDS
# ================================================================

model.eval()

with torch.no_grad():

    Y_pred_scaled = (
        model(
            X_valid_tensor
        )
        .cpu()
        .numpy()
    )


# ================================================================
# 13. CONVERT BACK TO ORIGINAL REWARD SCALE
# ================================================================

predicted_rewards = np.zeros(
    (
        len(validation_dataset),
        5
    ),
    dtype=np.float32
)

# Identity is exactly zero.
predicted_rewards[:, 0] = 0.0

# Convert the four standardized reward heads
# back to original reward units.
predicted_rewards[:, 1:] = (
    reward_scaler.inverse_transform(
        Y_pred_scaled[:, 1:]
    )
)


# ================================================================
# 14. POLICY ACTION
# ================================================================

predicted_action_id = np.argmax(
    predicted_rewards,
    axis=1
)


# ================================================================
# 15. ACTUAL REWARD OF SELECTED ACTION
# ================================================================

actual_rewards = (
    validation_dataset[
        REWARD_COLS
    ].to_numpy()
)

actual_policy_reward = (
    actual_rewards[
        np.arange(
            len(actual_rewards)
        ),
        predicted_action_id
    ]
)


# ================================================================
# 16. ORACLE
# ================================================================

oracle_reward = (
    actual_rewards.max(
        axis=1
    )
)


# ================================================================
# 17. PERFORMANCE
# ================================================================

identity_mean = (
    actual_rewards[:, 0]
    .mean()
)

policy_mean = (
    actual_policy_reward
    .mean()
)

oracle_mean = (
    oracle_reward
    .mean()
)

policy_gain = (
    policy_mean -
    identity_mean
)

oracle_gain = (
    oracle_mean -
    identity_mean
)


print()
print("=" * 80)
print("TARGET-STANDARDIZED POLICY PERFORMANCE")
print("=" * 80)

print(
    f"Identity mean reward : "
    f"{identity_mean:.6f}"
)

print(
    f"Policy mean reward   : "
    f"{policy_mean:.6f}"
)

print(
    f"Oracle mean reward   : "
    f"{oracle_mean:.6f}"
)

print()

print(
    f"Policy gain over Identity : "
    f"{policy_gain:.6f}"
)

print(
    f"Oracle gain over Identity  : "
    f"{oracle_gain:.6f}"
)

if oracle_gain > 0:

    gain_captured = (
        policy_gain /
        oracle_gain *
        100
    )

    print(
        f"Oracle gain captured       : "
        f"{gain_captured:.2f}%"
    )


# ================================================================
# 18. PREDICTED ACTION DISTRIBUTION
# ================================================================

ACTION_NAMES = {
    0: "Identity",
    1: "CLAHE",
    2: "Gamma",
    3: "Dehazing",
    4: "Denoising"
}


print()
print("=" * 80)
print("PREDICTED ACTION DISTRIBUTION")
print("=" * 80)

for action_id in range(5):

    count = np.sum(
        predicted_action_id == action_id
    )

    percentage = (
        count /
        len(predicted_action_id)
        *
        100
    )

    print(
        f"{action_id} "
        f"({ACTION_NAMES[action_id]:10s}) : "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )


# ================================================================
# 19. TRUE ORACLE ACTION DISTRIBUTION
# ================================================================

oracle_action_id = np.argmax(
    actual_rewards,
    axis=1
)


print()
print("=" * 80)
print("TRUE ORACLE ACTION DISTRIBUTION")
print("=" * 80)

for action_id in range(5):

    count = np.sum(
        oracle_action_id == action_id
    )

    percentage = (
        count /
        len(oracle_action_id)
        *
        100
    )

    print(
        f"{action_id} "
        f"({ACTION_NAMES[action_id]:10s}) : "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )


# ================================================================
# 20. PREDICTED REWARD MEANS
# ================================================================

print()
print("=" * 80)
print("PREDICTED REWARD MEANS")
print("=" * 80)

for action_id in range(5):

    print(
        f"{action_id} "
        f"({ACTION_NAMES[action_id]:10s}) : "
        f"{predicted_rewards[:, action_id].mean():.6f}"
    )


print()
print("=" * 80)
print("CELL 43 COMPLETED")
print("=" * 80)

EVALUATING TARGET-STANDARDIZED REWARD REGRESSION

Validation IQA rows : 10,000

VALIDATION DETECTION RESULT FILES
Identity   : 10,000
CLAHE      : 10,000
Gamma      : 10,000
Dehazing   : 10,000
Denoising  : 10,000

ACTUAL VALIDATION F1 TABLE
Rows : 10,000

FINAL VALIDATION DATASET
IQA rows              : 10,000
Actual F1 rows        : 10,000
Final merged rows     : 10,000
✓ Exact 10K validation dataset confirmed.

TARGET-STANDARDIZED POLICY PERFORMANCE
Identity mean reward : 0.000000
Policy mean reward   : -0.000689
Oracle mean reward   : 0.036502

Policy gain over Identity : -0.000689
Oracle gain over Identity  : 0.036502
Oracle gain captured       : -1.89%

PREDICTED ACTION DISTRIBUTION
0 (Identity  ) :  8,049 ( 80.49%)
1 (CLAHE     ) :    142 (  1.42%)
2 (Gamma     ) :  1,796 ( 17.96%)
3 (Dehazing  ) :     10 (  0.10%)
4 (Denoising ) :      3 (  0.03%)

TRUE ORACLE ACTION DISTRIBUTION
0 (Identity  ) :  3,900 ( 39.00%)
1 (CLAHE     ) :  2,358 ( 23.58%)
2 (Gamma     ) :  1,473 ( 14.73

In [1]:
# ============================================================
# STANDARD MLP — 20K FINAL TEST PATH CHECK
# ============================================================

from pathlib import Path

BASE_DIR = Path(r"E:\ML_Project")

SEARCH_DIRS = [
    BASE_DIR / "results" / "mlp_policy",
    BASE_DIR / "results",
]

print("=" * 80)
print("STANDARD MLP — 20K FINAL TEST FILE CHECK")
print("=" * 80)

print()

# ------------------------------------------------------------
# MLP MODEL / CHECKPOINT FILES
# ------------------------------------------------------------

print("MLP MODEL / CHECKPOINT FILES")
print("-" * 80)

model_files = []

for directory in SEARCH_DIRS:

    if directory.exists():

        for pattern in [
            "*.pt",
            "*.pth",
            "*.pkl",
            "*.joblib"
        ]:

            model_files.extend(
                directory.rglob(pattern)
            )

model_files = sorted(
    set(model_files)
)

if model_files:

    for path in model_files:
        print(path)

else:

    print("NONE FOUND")


# ------------------------------------------------------------
# 20K IQA / FEATURE FILES
# ------------------------------------------------------------

print()
print("20K / TEST IQA FEATURE FILES")
print("-" * 80)

feature_files = []

for directory in SEARCH_DIRS:

    if directory.exists():

        for path in directory.rglob(
            "*.csv"
        ):

            name = path.name.lower()

            if (
                "test" in name
                or "20k" in name
                or "iqa" in name
            ):

                feature_files.append(
                    path
                )

feature_files = sorted(
    set(feature_files)
)

if feature_files:

    for path in feature_files:
        print(path)

else:

    print("NONE FOUND")


# ------------------------------------------------------------
# FINAL TEST IMAGE DIRECTORIES
# ------------------------------------------------------------

print()
print("20K TEST IMAGE DIRECTORIES")
print("-" * 80)

for directory in [
    BASE_DIR / "data",
    BASE_DIR / "dataset",
    BASE_DIR / "datasets",
    BASE_DIR / "results",
]:

    if directory.exists():

        for path in directory.rglob("*"):

            if path.is_dir():

                name = path.name.lower()

                if (
                    "test" in name
                    or "20k" in name
                ):

                    print(path)

STANDARD MLP — 20K FINAL TEST FILE CHECK

MLP MODEL / CHECKPOINT FILES
--------------------------------------------------------------------------------
E:\ML_Project\results\final_analysis\cnn_policy\models\cnn_baseline_best.pt
E:\ML_Project\results\final_analysis\margin_policy_93feature\margin_policy_93feature_mlp.pt
E:\ML_Project\results\final_analysis\margin_policy_93feature\margin_policy_93feature_scaler.pkl
E:\ML_Project\results\policy_models\clahe_reward_lgbm.pkl
E:\ML_Project\results\policy_models\dehazing_reward_lgbm.pkl
E:\ML_Project\results\policy_models\denoising_reward_lgbm.pkl
E:\ML_Project\results\policy_models\gamma_reward_lgbm.pkl
E:\ML_Project\results\policy_models\model_info.pkl
E:\ML_Project\results\policy_models_high_confidence\clahe_reward_lgbm_high_confidence.pkl
E:\ML_Project\results\policy_models_high_confidence\dehazing_reward_lgbm_high_confidence.pkl
E:\ML_Project\results\policy_models_high_confidence\denoising_reward_lgbm_high_confidence.pkl
E:\ML_Project\res